[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB07_Optimizacion_Interoperabilidad_ONNX.ipynb)
# NB07 — Optimización, interoperabilidad y ONNX

**Continuidad NB04 → NB07 → NB08.** Usaremos la CNN de **piedra, papel o tijera** (128 × 128, píxeles en [0, 1], categorías `paper`, `rock`, `scissors`). La optimización con TFLite y la demostración de poda son experimentos; el archivo ONNX exporta el **modelo Keras original**, sin poda.


## 1. Preparación del entorno

**Antes de ejecutar:** instalaremos `tf2onnx` (conversión), `onnx` (validación del archivo) y `onnxruntime` (inferencia independiente de TensorFlow). `numpy`, `pandas` y `matplotlib` permiten procesar y presentar resultados. **Resultado esperado:** se muestran las versiones de TensorFlow y ONNX Runtime; todavía no se carga ni modifica ningún modelo.

In [ ]:
!pip -q install tf2onnx onnx onnxruntime

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import onnx
import onnxruntime as ort

from tensorflow import keras
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("ONNX Runtime:", ort.__version__)


## 2. Conectar Google Drive y conocer los archivos que crearemos

Al ejecutar la siguiente celda, Colab solicitará autorización para acceder a **Mi unidad** de Google Drive. **Autorizar el acceso no crea automáticamente las carpetas:** las celdas posteriores las crearán cuando sea necesario.

Este NB07 utiliza la carpeta **`Mi unidad/MachineLearning2026`** (sin espacio entre *Machine* y *Learning* y sin espacio antes de *2026*). Si no existe, se crea; si ya existe, se reutiliza.

```text
Mi unidad/
└── MachineLearning2026/
    ├── modelo_cnn_nb04.keras           ← CNN entrenada, guardada o recuperada
    └── implementacion/
        ├── modelo_cnn.onnx             ← modelo que utilizará NB08
        ├── modelo_quantized.tflite     ← experimento de cuantificación
        ├── class_names.json           ← orden de categorías
        └── model_contract.json        ← tamaño y preprocesamiento
```

**¿Cuándo aparecen?** El archivo `.keras` se guarda al entrenar por la **opción B**; en la **opción A** ya debe existir. La subcarpeta `implementacion` y sus cuatro archivos se crean al ejecutar las secciones **21 y 22**, no al conectar Drive.

**Importante:** al repetir las celdas de guardado pueden reemplazarse archivos con los mismos nombres. El notebook no modifica el NB04 ni publica nada en GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Elegir el recorrido: modelo del NB04 (opción A o B)

En NB04 entrenamos una CNN de **piedra, papel o tijera**, pero ese notebook no guardaba automáticamente el archivo `.keras`. Antes de continuar, identifica tu situación:

### Opción A — Ya tengo el modelo entrenado

1. Comprueba que el archivo `modelo_cnn_nb04.keras` esté en `Mi unidad/MachineLearning2026/`.
2. **Ejecuta la celda siguiente**: detectará y cargará el modelo existente, sin entrenarlo nuevamente.
3. Cuando aparezca **«Opción A: modelo cargado»**, **continúa en la sección 4: Contrato del modelo**. No debes ejecutar el entrenamiento de la opción B.

### Opción B — No tengo un modelo guardado

1. **Ejecuta la misma celda siguiente**. Si el archivo no existe, descargará los tres ZIP docentes y reconstruirá la arquitectura del NB04.
2. Espera a que termine el entrenamiento de **cinco épocas** y se guarde `modelo_cnn_nb04.keras` en Drive.
3. Cuando aparezca **«Opción B: entrenamiento finalizado y modelo guardado»**, **continúa en la sección 4**.

**No es necesario saltarse manualmente ninguna celda de código:** la celda decide el recorrido según la existencia del archivo. Así evitamos repetir entrenamiento o dejar variables sin inicializar. Si el archivo existente es incompatible, se detendrá con un error; **no lo sobrescribirá**.

| Parámetro | Valor | Función |
|---|---|---|
| `CLASS_NAMES` | `paper, rock, scissors` | Mantiene el orden de salida del NB04. |
| `IMG_SIZE` | `(128, 128)` | Tamaño de entrada de la CNN. |
| `SEED` | `42` | Ayuda a reproducir la partición y el entrenamiento. |
| `test_size` | `0.20` | Reserva 20 % de los datos de entrenamiento para validación. |
| `epochs` | `5` | Número de épocas en la opción B. |

**Atención:** reconstruir la arquitectura no recupera los pesos de una sesión anterior; la opción B obtiene **nuevos pesos entrenados**.

In [ ]:
from tensorflow.keras import layers
from zipfile import ZipFile
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split

CLASS_NAMES = ["paper", "rock", "scissors"]  # Orden exacto del NB04
IMG_SIZE = (128, 128)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
MODEL_PATH = "/content/drive/MyDrive/MachineLearning2026/modelo_cnn_nb04.keras"
Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)

if Path(MODEL_PATH).is_file():
    print("Opción A: cargando modelo guardado del NB04")
    model = keras.models.load_model(MODEL_PATH)
    print("Opción A: modelo cargado. Continúa en la sección 4.")
else:
    print("Opción B: reconstruyendo la CNN del NB04 con el dataset docente")
    raiz = Path("/content")
    base_url = ("https://raw.githubusercontent.com/juliopez/"
                "Taller-Fundamentos-Data-Science-Python/main/"
                "Machine_Learning_2026/02_Datasets/dataset_cnn/")
    for clase in ["paper", "rock", "scissors"]:
        nombre = f"dataset_cnn_{clase}.zip"
        zip_local = raiz / nombre
        urlretrieve(base_url + nombre, zip_local)
        with ZipFile(zip_local) as archivo:
            for elemento in archivo.infolist():
                partes = Path(elemento.filename)
                if partes.is_absolute() or ".." in partes.parts or partes.parts[0] != "dataset_cnn":
                    raise ValueError(f"Ruta no válida en ZIP: {elemento.filename}")
            archivo.extractall(raiz)
    train_dir = raiz / "dataset_cnn" / "train"
    rutas, etiquetas = [], []
    for indice, clase in enumerate(CLASS_NAMES):
        archivos = sorted((train_dir / clase).glob("*.png"))
        assert archivos, f"No se encontraron imágenes para {clase}"
        rutas.extend(str(p) for p in archivos)
        etiquetas.extend([indice] * len(archivos))
    rutas_train, rutas_val, etiquetas_train, etiquetas_val = train_test_split(
        rutas, etiquetas, test_size=0.20, random_state=SEED, stratify=etiquetas
    )
    def leer(ruta, etiqueta):
        imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
        imagen = tf.image.resize(imagen, IMG_SIZE)
        return tf.cast(imagen, tf.float32) / 255.0, etiqueta
    def dataset(rutas_, etiquetas_, mezclar=False):
        ds = tf.data.Dataset.from_tensor_slices((rutas_, np.array(etiquetas_, dtype=np.int32)))
        if mezclar:
            ds = ds.shuffle(len(rutas_), seed=SEED, reshuffle_each_iteration=True)
        return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(16).prefetch(tf.data.AUTOTUNE)
    model = keras.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(16, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(32, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ], name="cnn_piedra_papel_tijera")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(dataset(rutas_train, etiquetas_train, True),
              validation_data=dataset(rutas_val, etiquetas_val), epochs=5)
    model.save(MODEL_PATH)
    print("Opción B: entrenamiento finalizado y modelo guardado:", MODEL_PATH)
    print("Continúa en la sección 4.")

assert tuple(model.input_shape[1:]) == (128, 128, 3), "El modelo no coincide con NB04"
assert model.output_shape[-1] == 3, "Se esperan tres categorías"
model.summary()

## 4. Contrato del modelo: entrada, clases y preprocesamiento

El **contrato** describe exactamente qué espera recibir la CNN. En NB04 cada imagen se convertía a RGB, se redimensionaba a **128 × 128** y sus píxeles se dividían entre **255**. **No** se utilizó `MobileNetV2.preprocess_input`: aplicarlo aquí produciría entradas incompatibles con el entrenamiento.

El orden de salida es **paper, rock, scissors**. ONNX y Streamlit deben conservar este mismo contrato.

**¿Por qué es importante?** Un modelo exportado no conoce por sí mismo el nombre de cada categoría ni cómo preparábamos las imágenes. Si la aplicación cambia el tamaño, el rango de píxeles o el orden de las clases, puede producir resultados incorrectos aunque el archivo ONNX sea válido. **Resultado esperado:** un diccionario con las reglas que NB08 debe reproducir.

In [ ]:
contrato = {
    "input_shape": [1, 128, 128, 3],
    "tipo_entrada": "float32",
    "canales": "RGB",
    "preprocesamiento": "RGB, resize 128x128, float32 / 255.0",
    "clases": CLASS_NAMES
}
print(contrato)

## 5. Seleccionar una imagen de prueba

Utilizaremos una imagen **del conjunto de prueba original del NB04**, diferente de las usadas para entrenar. Si cargaste el modelo guardado (opción A), el dataset se descargará igualmente para disponer de una imagen de ejemplo.

**Antes de ejecutar:** la celda descarga el dataset docente solo cuando falta en Colab y selecciona una imagen `rock` de prueba. El redimensionamiento permite visualizarla y utilizarla posteriormente. **Resultado esperado:** aparece la imagen y su categoría real; aún no se genera ninguna predicción.

In [ ]:
from PIL import Image
from zipfile import ZipFile
from urllib.request import urlretrieve

TEST_DIR = Path("/content/dataset_cnn/test")
if not TEST_DIR.is_dir():
    raiz = Path("/content")
    base_url = ("https://raw.githubusercontent.com/juliopez/"
                "Taller-Fundamentos-Data-Science-Python/main/"
                "Machine_Learning_2026/02_Datasets/dataset_cnn/")
    for clase in CLASS_NAMES:
        nombre = f"dataset_cnn_{clase}.zip"
        zip_local = raiz / nombre
        urlretrieve(base_url + nombre, zip_local)
        with ZipFile(zip_local) as archivo:
            for elemento in archivo.infolist():
                partes = Path(elemento.filename)
                if partes.is_absolute() or ".." in partes.parts or partes.parts[0] != "dataset_cnn":
                    raise ValueError(f"Ruta no válida: {elemento.filename}")
            archivo.extractall(raiz)
IMAGE_PATH = sorted((TEST_DIR / "rock").glob("*.png"))[0]
img = Image.open(IMAGE_PATH).convert("RGB").resize(IMG_SIZE)
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"Ejemplo de prueba: {IMAGE_PATH.parent.name}")
plt.axis("off")
plt.show()

## 6. Preprocesar la imagen como en NB04

**RGB → 128 × 128 → float32 → dividir entre 255 → agregar dimensión de lote**. La entrada resultante tendrá forma `(1, 128, 128, 3)`. Usaremos exactamente el **mismo tensor `x`** para Keras, TFLite y ONNX.

In [ ]:
x = np.asarray(img, dtype=np.float32)[None, ...] / 255.0
assert x.shape == (1, 128, 128, 3)
print("Shape:", x.shape, "dtype:", x.dtype)
print("Rango de píxeles:", float(x.min()), float(x.max()))

## 7. Baseline del modelo original

Antes de optimizar o convertir el modelo registraremos:

- tamaño;
- latencia de inferencia;
- predicción;
- confianza.

Estos valores constituyen la **baseline**.

**¿Qué mediremos y por qué?**

| Medida | Interpretación |
|---|---|
| Tamaño (MB) | Espacio ocupado por el archivo Keras en disco. |
| Latencia (ms) | Tiempo de una inferencia con un lote de una imagen; **menor** suele ser mejor. |
| Clase predicha | Categoría con la probabilidad más alta. |
| Confianza | Mayor salida softmax; **no garantiza** que la predicción sea correcta. |

El código realiza **5 ejecuciones de calentamiento** (*warm-up*) y **30 inferencias cronometradas**. Estas mediciones son orientativas: Keras y ONNX Runtime pueden tener sobrecostos distintos y la máquina de Colab puede variar. **Resultado esperado:** una línea base (*baseline*) con tamaño, latencia, categoría y confianza; luego compararemos versiones usando la misma imagen.

### ¿Qué es la latencia y por qué nos importa?

La **latencia de inferencia** es el tiempo transcurrido desde que entregamos al modelo una entrada ya preparada hasta que obtenemos su predicción. Se expresa normalmente en **milisegundos (ms)**: **20 ms = 0,020 segundos**.

**Analogía:** en una cafetería, el tiempo que tarda la máquina en preparar el café una vez recibida la orden se parece a la latencia del modelo. El tiempo total que espera el cliente también incluye hacer la fila y pagar. Del mismo modo, el tiempo total que percibe el usuario de nuestra aplicación incluye **subir la imagen, redimensionarla, ejecutar el modelo y mostrar el resultado**; la latencia que medimos aquí **no abarca necesariamente todas esas etapas**.

**¿Por qué importa?** Una aplicación interactiva necesita responder oportunamente. Si dos modelos tienen exactitud similar, pero uno requiere menos tiempo y recursos para inferir, podría ser más adecuado para un teléfono o un servicio con muchas solicitudes.

**¿Cómo la medimos?** Primero realizamos **cinco ejecuciones de calentamiento** (*warm-up*) para evitar que la inicialización inicial distorsione la comparación. Luego cronometramos **30 ejecuciones** y calculamos su promedio.

**¿Qué debemos observar?** Menos milisegundos suele significar una inferencia más rápida, **no una clasificación más correcta**. Además, Keras, TFLite y ONNX Runtime pueden tener distintos costos de llamada: sus tiempos en este notebook son **orientativos**, no una prueba definitiva de superioridad. Para comparar rigurosamente, debemos mantener el mismo hardware, tamaño de lote, entradas y condiciones de ejecución.

In [ ]:
keras_size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)

# Warm-up
for _ in range(5):
    _ = model.predict(x, verbose=0)

tiempos = []

for _ in range(30):
    inicio = time.perf_counter()
    pred = model.predict(x, verbose=0)
    fin = time.perf_counter()
    tiempos.append((fin - inicio) * 1000)

keras_latency = np.mean(tiempos)
keras_prediction = int(np.argmax(pred[0]))
keras_confidence = float(np.max(pred[0]))

print(f"Tamaño modelo Keras: {keras_size_mb:.2f} MB")
print(f"Latencia promedio: {keras_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[keras_prediction])
print(f"Confianza: {keras_confidence:.2%}")


## 8. Cuantificación: reducir precisión para intentar usar menos recursos

### Analogía: una fotografía con menos niveles de color

Una fotografía puede almacenar muchos matices de color. Si representamos esos matices con menos niveles, el archivo puede ocupar menos espacio y seguir viéndose muy parecido, aunque se pierde precisión. **Cuantificar una red neuronal es parecido:** representamos algunos números (por ejemplo, pesos) con menos bits.

Por ejemplo, **`float32` utiliza 32 bits** para representar un valor, mientras que **`int8` utiliza 8 bits**. Esto **no significa** que toda conversión TFLite pase automáticamente a int8 ni que su archivo final mida exactamente la cuarta parte: hay otros datos, estructuras y operadores.

### ¿Siempre debemos cuantificar?

**No.** Es una decisión de optimización, útil especialmente cuando importan el almacenamiento, la memoria o la ejecución en dispositivos con recursos limitados.

Antes de cuantificar, pregúntate:

1. **¿Qué problema quiero resolver?** ¿Tamaño, memoria o velocidad?
2. **¿Qué representación acepta mi dispositivo?** No todos los motores aceleran los mismos tipos numéricos.
3. **¿Qué cambia en las predicciones?** Debemos comparar con la misma colección de imágenes.
4. **¿El beneficio es real?** Menor tamaño **no garantiza** menor latencia ni igual exactitud.

### ¿Qué hace este notebook?

El código usa `tf.lite.Optimize.DEFAULT`, una **optimización por defecto de TFLite** que puede cuantificar pesos de forma dinámica. No hemos configurado un conjunto representativo ni una cuantificación entera completa de entradas y salidas. **Verificaremos el tipo de entrada/salida del archivo generado y mediremos tamaño, latencia y predicciones**, en lugar de asumir que todos sus números son int8.

**Objetivo:** comprender el intercambio entre **eficiencia** y **fidelidad de las predicciones**, conservando intacto el modelo Keras original.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

TFLITE_PATH = "/content/modelo_quantized.tflite"

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_quant_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 ** 2)

print(f"Tamaño TFLite cuantificado: {tflite_size_mb:.2f} MB")


## 9. Inferencia con el modelo cuantificado

**Antes de ejecutar:** `tf.lite.Interpreter` carga el archivo TFLite, `allocate_tensors()` prepara memoria y `get_input_details()` / `get_output_details()` muestran las formas y tipos exigidos. **Comprueba** que la entrada sea compatible con una imagen de 128 × 128 y tres canales. El código de la sección siguiente adaptará el tipo del tensor; el rango de valores debe conservar el preprocesamiento del NB04.

**Cómo leer los resultados:** se mostrará la forma de entrada y salida y el tipo de dato de cada tensor. Una forma o tipo inesperados indican que debemos revisar el contrato antes de ejecutar inferencias.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Entrada:", input_details[0]["shape"], input_details[0]["dtype"])
print("Salida:", output_details[0]["shape"], output_details[0]["dtype"])


In [ ]:
tflite_input = x.astype(input_details[0]["dtype"])

# Warm-up
for _ in range(5):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)
    interpreter.invoke()

tiempos_tflite = []

for _ in range(30):
    interpreter.set_tensor(input_details[0]["index"], tflite_input)

    inicio = time.perf_counter()
    interpreter.invoke()
    fin = time.perf_counter()

    tiempos_tflite.append((fin - inicio) * 1000)

tflite_pred = interpreter.get_tensor(output_details[0]["index"])

tflite_latency = np.mean(tiempos_tflite)
tflite_prediction = int(np.argmax(tflite_pred[0]))
tflite_confidence = float(np.max(tflite_pred[0]))

print(f"Latencia TFLite: {tflite_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[tflite_prediction])
print(f"Confianza: {tflite_confidence:.2%}")


## 10. Comparar modelo original y cuantificado

**Qué compararemos:** tamaño, latencia, clase y confianza del **mismo ejemplo**. Una reducción del archivo no demuestra por sí sola que el modelo sea más rápido o más preciso. La tabla permite formular hipótesis, pero para evaluar calidad predictiva se necesitaría revisar múltiples imágenes.

In [ ]:
comparacion_optimizacion = pd.DataFrame({
    "Modelo": ["Keras original", "TFLite cuantificado"],
    "Tamaño_MB": [keras_size_mb, tflite_size_mb],
    "Latencia_ms": [keras_latency, tflite_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[tflite_prediction]
    ],
    "Confianza": [
        keras_confidence,
        tflite_confidence
    ]
})

comparacion_optimizacion


## 11. Poda de redes: ¿es obligatorio eliminar conexiones?

### Analogía: podar un árbol

Al podar un árbol retiramos ramas que consideramos prescindibles para conservar una estructura manejable. Pero **si cortamos demasiadas ramas, podemos perjudicarlo**. En una red neuronal sucede algo parecido: algunos pesos pueden convertirse en cero sin cambiar demasiado las predicciones; otros son importantes.

La **poda** (*pruning*) busca reducir la cantidad de conexiones o pesos activos. En nuestro experimento utilizaremos **poda por magnitud**: los pesos con valor absoluto más pequeño son candidatos a convertirse en cero.

**¿Debemos podar siempre? No.** Es una técnica opcional que se justifica cuando necesitamos optimizar memoria, almacenamiento o cómputo, y cuando el sistema de ejecución realmente puede aprovechar esa estructura.

### ¿Qué debemos considerar antes de podar?

1. **Objetivo:** ¿qué recurso queremos ahorrar?
2. **Criterio:** aquí usamos la **magnitud** del peso, una aproximación sencilla; magnitud pequeña **no demuestra** irrelevancia.
3. **Intensidad:** una poda mayor puede alterar más las predicciones.
4. **Calidad:** hay que comparar el rendimiento sobre **varias imágenes**, idealmente con exactitud sobre un conjunto de prueba.
5. **Beneficio real:** introducir ceros **no reduce automáticamente** el tamaño del archivo ni la latencia; depende del formato y del motor.

**¿Qué es *sparsity*?** Es la **proporción de parámetros exactamente iguales a cero**. El siguiente código mide primero la sparsity del modelo original. **No modifica sus pesos.**

In [ ]:
pesos = model.get_weights()

total = 0
ceros_originales = 0

for w in pesos:
    total += w.size
    ceros_originales += np.count_nonzero(w == 0)

sparsity_original = ceros_originales / total

print(f"Sparsity original: {sparsity_original:.2%}")


## 12. Experimentar con distintos niveles de poda

En lugar de asumir que un único nivel es el mejor, compararemos **10 %, 30 % y 50 %** de los pesos candidatos a poda por magnitud. Para cada nivel calcularemos el umbral, la *sparsity* global y la predicción sobre la imagen de ejemplo.

**Importante:** los porcentajes se aplican al conjunto de **pesos de tensores multidimensionales** (por ejemplo, convoluciones y capas densas), no a los sesgos. Por ello, la *sparsity* global puede ser distinta de 10 %, 30 % o 50 %.

El nivel **30 %** quedará como ejemplo para la copia podada de la sección 13. **Ninguno de estos experimentos modifica el modelo Keras original.**

**¿Qué observar?** Si aumentan los ceros, pero la clase predicha cambia o la confianza se altera demasiado, debemos investigar. Incluso si una imagen conserva su clase, **no podemos concluir** que el rendimiento general sea igual. Tampoco se obtiene automáticamente un archivo más pequeño.

In [ ]:
# Comparamos tres intensidades de poda sin alterar el modelo original.
pesos = model.get_weights()
valores_absolutos = np.concatenate([
    np.abs(w.ravel()) for w in pesos if w.ndim > 1
])
resultados_poda = []
pesos_por_nivel = {}

for porcentaje in (10, 30, 50):
    umbral_nivel = np.percentile(valores_absolutos, porcentaje)
    pesos_nivel = []
    for w in pesos:
        copia = w.copy()
        if copia.ndim > 1:
            copia[np.abs(copia) < umbral_nivel] = 0
        pesos_nivel.append(copia)
    pesos_por_nivel[porcentaje] = pesos_nivel
    copia_modelo = keras.models.clone_model(model)
    copia_modelo.set_weights(pesos_nivel)
    probabilidades = copia_modelo.predict(x, verbose=0)[0]
    total = sum(w.size for w in pesos_nivel)
    ceros = sum(np.count_nonzero(w == 0) for w in pesos_nivel)
    resultados_poda.append({
        "Poda candidata (%)": porcentaje,
        "Sparsity global (%)": round(100 * ceros / total, 2),
        "Clase predicha": CLASS_NAMES[int(np.argmax(probabilidades))],
        "Confianza (%)": round(100 * float(np.max(probabilidades)), 2)
    })

display(pd.DataFrame(resultados_poda))
# Mantenemos el ejemplo del 30 % para la siguiente sección.
pesos_podados = pesos_por_nivel[30]
umbral = np.percentile(valores_absolutos, 30)
total_podado = sum(w.size for w in pesos_podados)
ceros_podados = sum(np.count_nonzero(w == 0) for w in pesos_podados)
sparsity_podada = ceros_podados / total_podado
print(f"Umbral del ejemplo (30 %): {umbral:.6f}")
print(f"Sparsity global con poda del 30 %: {sparsity_podada:.2%}")

## 13. Crear una copia del modelo con pesos podados

Esta versión se utiliza solamente para comparar el efecto de la poda.

**Antes de ejecutar:** `clone_model` crea otra instancia con la misma arquitectura y `set_weights` le asigna los pesos podados. El modelo original permanece intacto. Compararemos la clase y la confianza de **la misma imagen**. Una coincidencia de clase no demuestra que la poda conserve el rendimiento en todo el dataset.

In [ ]:
pruned_model = keras.models.clone_model(model)
pruned_model.set_weights(pesos_podados)

pred_pruned = pruned_model.predict(x, verbose=0)

pruned_prediction = int(np.argmax(pred_pruned[0]))
pruned_confidence = float(np.max(pred_pruned[0]))

print("Clase original:", CLASS_NAMES[keras_prediction])
print("Clase después de poda:", CLASS_NAMES[pruned_prediction])
print(f"Confianza original: {keras_confidence:.2%}")
print(f"Confianza después de poda: {pruned_confidence:.2%}")


## 14. Interpretar la poda

**Conclusión principal:** podar más **no significa optimizar mejor**. La poda por magnitud es una demostración didáctica y puede afectar la calidad predictiva. Para decidir si vale la pena, necesitamos medir el desempeño en un conjunto de prueba, y comprobar el tamaño y la latencia del modelo realmente desplegado.

**Preguntas:** ¿aumentó la sparsity al pasar de 10 % a 50 %? ¿Cambió la clase predicha? ¿Podemos asegurar que la exactitud se conserva a partir de una imagen? ¿Por qué una red con más ceros puede seguir ocupando prácticamente el mismo espacio?

## 15. Exportar el modelo a ONNX

ONNX permite representar el modelo en un formato interoperable para ejecutarlo posteriormente mediante runtimes diferentes del framework original.


**Atención:** exportamos la **CNN original del NB04**. Las versiones cuantificada y podada son comparaciones didácticas independientes. Para mantener el contrato, la entrada ONNX será `float32` con forma `(None, 128, 128, 3)` y valores entre 0 y 1.

**Parámetros de exportación**

| Parámetro | Valor | Función |
|---|---|---|
| `TensorSpec` | `(None, 128, 128, 3)` | Define la forma de entrada; `None` admite distintos tamaños de lote. |
| `tf.float32` | 32 bits | Tipo numérico esperado por el modelo original. |
| `opset` | `13` | Versión del conjunto de operadores ONNX solicitada al convertidor. |
| `ONNX_PATH` | `/content/modelo_cnn.onnx` | Archivo que se generará para ejecutar el modelo con ONNX Runtime. |

**Qué hace el código:** envuelve la inferencia de Keras en una función TensorFlow y la convierte a ONNX. **No vuelve a entrenar** ni incorpora los pesos de la copia podada. **Resultado esperado:** un archivo `.onnx` en Colab. La conversión depende de la compatibilidad entre versiones instaladas; la validaremos en los pasos siguientes.

In [ ]:
import tf2onnx

ONNX_PATH = "/content/modelo_cnn.onnx"
input_signature = (tf.TensorSpec((None, 128, 128, 3), tf.float32, name="input"),)

# tf2onnx.from_keras puede presentar incompatibilidades con ciertas versiones de Keras 3.
# Exportamos primero una función TensorFlow concreta y la convertimos a ONNX.
@tf.function(input_signature=list(input_signature))
def inferencia_tf(entrada):
    return {"probabilidades": model(entrada, training=False)}

onnx_model, _ = tf2onnx.convert.from_function(
    inferencia_tf, input_signature=list(input_signature), opset=13
)
with open(ONNX_PATH, "wb") as archivo:
    archivo.write(onnx_model.SerializeToString())
print("Modelo ONNX generado:", ONNX_PATH)

## 16. Validar el archivo ONNX

**Antes de ejecutar:** `onnx.load` abre el archivo y `onnx.checker.check_model` comprueba su estructura. Esto detecta problemas de formato, **pero no demuestra por sí solo equivalencia predictiva**. **Resultado esperado:** confirmación de validez estructural y tamaño del archivo.

In [ ]:
onnx_model_check = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model_check)

onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)

print("Modelo ONNX válido")
print(f"Tamaño: {onnx_size_mb:.2f} MB")


## 17. Cargar ONNX con ONNX Runtime

**Antes de ejecutar:** `ort.InferenceSession` crea una sesión para ejecutar ONNX en CPU. Los nombres de entrada y salida se obtienen del propio archivo: **no debemos inventarlos**. **Resultado esperado:** nombres y formas de los tensores, coherentes con el contrato de la sección 4.

In [ ]:
session = ort.InferenceSession(
    ONNX_PATH,
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Input:", input_name)
print("Input shape:", session.get_inputs()[0].shape)
print("Output:", output_name)
print("Output shape:", session.get_outputs()[0].shape)


## 18. Inferencia mediante ONNX Runtime

El modelo exportado debe recibir exactamente el mismo tipo de entrada que el modelo original.

**Qué hace el código:** envía exactamente el mismo tensor `x` utilizado por Keras al motor ONNX Runtime. Realiza 5 ejecuciones de calentamiento y 30 inferencias cronometradas. **Resultado esperado:** latencia media, clase y confianza. Las diferencias de latencia pueden depender tanto del runtime como del hardware y de cómo se mide.

In [ ]:
onnx_input = x.astype(np.float32)

# Warm-up
for _ in range(5):
    _ = session.run(
        [output_name],
        {input_name: onnx_input}
    )

tiempos_onnx = []

for _ in range(30):
    inicio = time.perf_counter()

    onnx_pred = session.run(
        [output_name],
        {input_name: onnx_input}
    )[0]

    fin = time.perf_counter()
    tiempos_onnx.append((fin - inicio) * 1000)

onnx_latency = np.mean(tiempos_onnx)
onnx_prediction = int(np.argmax(onnx_pred[0]))
onnx_confidence = float(np.max(onnx_pred[0]))

print(f"Latencia ONNX Runtime: {onnx_latency:.2f} ms")
print("Clase predicha:", CLASS_NAMES[onnx_prediction])
print(f"Confianza: {onnx_confidence:.2%}")


## 19. Comparar Keras y ONNX

**Cómo interpretar la tabla:** Keras y ONNX ejecutan la misma red y deberían producir predicciones muy próximas, aunque pueden existir pequeñas diferencias numéricas. Compara también tamaños y latencias, sin asumir que ONNX será siempre más rápido.

In [ ]:
comparacion_onnx = pd.DataFrame({
    "Modelo": ["Keras", "ONNX"],
    "Tamaño_MB": [keras_size_mb, onnx_size_mb],
    "Latencia_ms": [keras_latency, onnx_latency],
    "Clase_predicha": [
        CLASS_NAMES[keras_prediction],
        CLASS_NAMES[onnx_prediction]
    ],
    "Confianza": [
        keras_confidence,
        onnx_confidence
    ]
})

comparacion_onnx


## 20. Diferencia absoluta máxima: ¿Keras y ONNX calculan casi lo mismo?

### Analogía: dos calculadoras resolviendo la misma operación

Si dos calculadoras producen `0,850` y `0,849`, sus respuestas son **muy próximas**, aunque no idénticas. Al convertir nuestra CNN de Keras a ONNX, esperamos que las **probabilidades de las tres clases** sigan siendo prácticamente iguales.

Para cada categoría calculamos:

**Diferencia absoluta = |probabilidad Keras − probabilidad ONNX|**

Después elegimos **la mayor** de esas tres diferencias. Eso es la **diferencia absoluta máxima**.

| Clase (ejemplo) | Keras | ONNX | Diferencia absoluta |
|---|---:|---:|---:|
| rock | 0,850 | 0,849 | 0,001 |
| paper | 0,100 | 0,101 | 0,001 |
| scissors | 0,050 | 0,050 | 0,000 |

La diferencia máxima es **0,001**, equivalente a **0,1 puntos porcentuales** de probabilidad.

### ¿En qué debemos prestar atención?

- **Cerca de cero:** buena señal de fidelidad numérica de la conversión.
- **Diferencia apreciable:** revisar preprocesamiento, operadores y conversión.
- **Cambio de clase predicha:** requiere especial atención, incluso si la diferencia es pequeña, porque las dos probabilidades mayores pueden estar muy próximas.

Usaremos **0,001 como tolerancia orientativa didáctica**, **no como un límite universal**. El código mostrará una tabla por clase y advertirá si la diferencia supera esa referencia.

**Importante:** que Keras y ONNX coincidan **no significa que acierten**. Pueden producir los mismos resultados y equivocarse ambos. Además, la comprobación de una sola imagen no demuestra equivalencia en todo el dataset.

In [ ]:
probabilidades_keras = np.asarray(pred[0], dtype=np.float64)
probabilidades_onnx = np.asarray(onnx_pred[0], dtype=np.float64)
diferencias = np.abs(probabilidades_keras - probabilidades_onnx)
diferencia_maxima = float(np.max(diferencias))
TOLERANCIA_DIDACTICA = 0.001

tabla_equivalencia = pd.DataFrame({
    "Categoría": CLASS_NAMES,
    "Keras": probabilidades_keras,
    "ONNX": probabilidades_onnx,
    "Diferencia absoluta": diferencias
})
display(tabla_equivalencia)
print(f"Diferencia absoluta máxima: {diferencia_maxima:.8f}")
print("Tolerancia orientativa:", TOLERANCIA_DIDACTICA)
if keras_prediction != onnx_prediction:
    print("ATENCIÓN: Keras y ONNX predicen clases distintas. Revisar la conversión.")
elif diferencia_maxima > TOLERANCIA_DIDACTICA:
    print("REVISAR: diferencia superior a la tolerancia didáctica; investigar antes de desplegar.")
else:
    print("Resultado próximo en esta imagen; verificar también con más imágenes.")

### 20.1. Comparación visual: tres formatos, las mismas imágenes

Como en NB06, no nos limitaremos a una tabla de métricas. Seleccionaremos **dos imágenes de prueba por categoría** (`paper`, `rock`, `scissors`) y compararemos las predicciones de **Keras original, TFLite optimizado y ONNX**.

Las tres versiones reciben exactamente **la misma imagen preprocesada** (RGB, 128 × 128, valores entre 0 y 1). Para cada imagen veremos la **etiqueta real**, las tres clases predichas y sus confianzas.

**Cómo interpretar los resultados:** una diferencia entre Keras y ONNX merece revisar la exportación; TFLite puede mostrar pequeñas variaciones por optimización. Que los tres coincidan en seis imágenes **no prueba** que su exactitud general sea idéntica. La copia con poda no se incluye aquí porque es un experimento distinto y no constituye un formato exportado.

In [ ]:
# Dos imágenes de prueba por categoría, sin volver a entrenar.
rutas_comparacion = [
    ruta
    for clase in CLASS_NAMES
    for ruta in sorted((TEST_DIR / clase).glob("*.png"))[:2]
]
assert len(rutas_comparacion) == 6, "Se esperan dos imágenes de prueba por cada categoría."

def preparar_para_modelos(ruta):
    with Image.open(ruta) as imagen:
        arreglo = np.asarray(imagen.convert("RGB").resize(IMG_SIZE), dtype=np.float32)
    return arreglo[None, ...] / 255.0

def predecir_tflite(entrada):
    detalles = input_details[0]
    # El contrato habitual de este ejercicio utiliza float32; se comprueba
    # el tipo para no tratar erróneamente una entrada cuantificada como float.
    if np.issubdtype(detalles["dtype"], np.integer):
        escala, cero = detalles["quantization"]
        assert escala > 0, "Falta la escala para cuantizar la entrada TFLite."
        entrada = np.rint(entrada / escala + cero)
        entrada = np.clip(entrada, np.iinfo(detalles["dtype"]).min,
                          np.iinfo(detalles["dtype"]).max).astype(detalles["dtype"])
    else:
        entrada = entrada.astype(detalles["dtype"])
    interpreter.set_tensor(detalles["index"], entrada)
    interpreter.invoke()
    salida = interpreter.get_tensor(output_details[0]["index"])[0]
    if np.issubdtype(output_details[0]["dtype"], np.integer):
        escala, cero = output_details[0]["quantization"]
        salida = (salida.astype(np.float32) - cero) * escala
    return salida

filas_comparacion = []
fig, ejes = plt.subplots(2, 3, figsize=(11, 7))
for eje, ruta in zip(ejes.flat, rutas_comparacion):
    entrada = preparar_para_modelos(ruta)
    probabilidades = {
        "Keras": model.predict(entrada, verbose=0)[0],
        "TFLite": predecir_tflite(entrada),
        "ONNX": session.run([output_name], {input_name: entrada})[0][0]
    }
    fila = {"Imagen": ruta.name, "Clase real": ruta.parent.name,
            "Diferencia máx. Keras-ONNX": float(np.max(np.abs(
                probabilidades["Keras"] - probabilidades["ONNX"]))) }
    for nombre, probs in probabilidades.items():
        indice = int(np.argmax(probs))
        fila[nombre] = CLASS_NAMES[indice]
        fila[nombre + " confianza"] = f"{float(probs[indice]):.1%}"
    filas_comparacion.append(fila)
    with Image.open(ruta) as imagen:
        eje.imshow(imagen.convert("RGB"))
    eje.set_title(f"Real: {ruta.parent.name}\n"
                  f"Keras: {fila['Keras']} | ONNX: {fila['ONNX']}", fontsize=9)
    eje.axis("off")
plt.tight_layout()
plt.show()
display(pd.DataFrame(filas_comparacion))

**Preguntas para interpretar la comparación:** ¿coinciden las tres predicciones en cada imagen? ¿En qué casos cambia la confianza? ¿Coincidir en la etiqueta significa que las probabilidades son idénticas? ¿Qué información adicional necesitaríamos para afirmar que la optimización conserva el rendimiento?

**Importante:** estas imágenes pertenecen al conjunto de prueba del NB04. Aquí se usan para **demostrar y verificar la inferencia** entre formatos, no para elegir repetidamente una nueva arquitectura o ajustar hiperparámetros.

**Verificación adicional de equivalencia:** la siguiente celda resume también la **diferencia absoluta máxima Keras–ONNX por imagen**, no solo si coincide la clase. Una diferencia pequeña en varias imágenes aumenta nuestra confianza en la conversión, pero no constituye una prueba exhaustiva.

## 21. Guardar los artefactos para NB08

Copiaremos a Google Drive el **modelo ONNX original**, el modelo TFLite de comparación y los metadatos del contrato. En **NB08** utilizaremos `modelo_cnn.onnx` y `class_names.json` para construir la aplicación Streamlit.

**Qué guardaremos:** `modelo_cnn.onnx` es el artefacto principal para NB08; `modelo_quantized.tflite` se conserva para comparar formatos. El directorio `implementacion` estará dentro de Google Drive. **Resultado esperado:** ambos archivos copiados sin alterar la CNN de origen.

In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MachineLearning2026/implementacion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import shutil

shutil.copy(ONNX_PATH, OUTPUT_DIR / "modelo_cnn.onnx")
shutil.copy(TFLITE_PATH, OUTPUT_DIR / "modelo_quantized.tflite")

print("Archivos guardados en:")
print(OUTPUT_DIR)


## 22. Guardar el orden de las clases y el contrato de inferencia

No basta con guardar los pesos: la aplicación debe conocer **el orden de las categorías y el preprocesamiento exacto**. Guardaremos ambos metadatos junto al archivo ONNX.

**Resultado esperado:** `class_names.json` contiene `paper`, `rock`, `scissors` en el orden correcto; `model_contract.json` registra la entrada RGB de 128 × 128 normalizada entre 0 y 1. NB08 necesitará respetar ambos, aunque el contrato no se cargue automáticamente desde ese archivo.

In [ ]:
import json
CLASSES_PATH = OUTPUT_DIR / "class_names.json"
CONTRACT_PATH = OUTPUT_DIR / "model_contract.json"
with open(CLASSES_PATH, "w", encoding="utf-8") as archivo:
    json.dump(CLASS_NAMES, archivo, ensure_ascii=False, indent=2)
with open(CONTRACT_PATH, "w", encoding="utf-8") as archivo:
    json.dump(contrato, archivo, ensure_ascii=False, indent=2)
print("Clases:", CLASSES_PATH)
print("Contrato:", CONTRACT_PATH)

## 23. Comparación final

Complete la tabla con las métricas obtenidas para las versiones que correspondan a su proyecto.

**Cómo interpretar la comparación final:** observa las diferencias entre formatos y separa las conclusiones de **tamaño**, **latencia** y **predicción**. No afirmes que un formato es mejor en todos los aspectos a partir de una sola imagen o una única sesión de Colab.

In [ ]:
comparacion_final = pd.DataFrame({
    "Versión": [
        "Keras original",
        "TFLite cuantificado",
        "ONNX"
    ],
    "Tamaño_MB": [
        keras_size_mb,
        tflite_size_mb,
        onnx_size_mb
    ],
    "Latencia_ms": [
        keras_latency,
        tflite_latency,
        onnx_latency
    ],
    "Confianza_imagen_prueba": [
        keras_confidence,
        tflite_confidence,
        onnx_confidence
    ]
})

comparacion_final


## 24. Actividad

A partir de la CNN de piedra, papel o tijera del NB04:

1. Registre tamaño y latencia del modelo original.
2. Defina una baseline antes de optimizar.
3. Genere una versión cuantificada.
4. Compare tamaño, latencia y predicción con el modelo original.
5. Analice conceptualmente la poda y mida la sparsity obtenida.
6. Determine qué efectos podría tener una poda excesiva.
7. Exporte el modelo a ONNX.
8. Valide la estructura del archivo ONNX.
9. Ejecute inferencia mediante ONNX Runtime.
10. Compare la predicción de Keras y ONNX.
11. Compruebe que el contrato de entrada y salida se conserva.
12. Justifique qué versión utilizaría para la implementación del producto final.

**Reflexión guiada:** ¿por qué la versión podada no se exportó como modelo final? ¿Qué diferencia existe entre reducir el tamaño de un archivo y acelerar la inferencia? ¿Por qué es necesario guardar las clases y el preprocesamiento junto al modelo?

## 25. Cierre y continuidad del aprendizaje

Este notebook inicia la etapa técnica del proceso de integración y despliegue.

El resultado esperado es disponer de evidencia sobre eficiencia e interoperabilidad y de un modelo preparado para integrarse posteriormente con una aplicación.

**Modelo definitivo → baseline → cuantificación/poda → comparación → ONNX → ONNX Runtime → modelo preparado para implementación**

En NB08 se utilizarán estos artefactos para construir la aplicación con **Streamlit**, organizar el proyecto en **GitHub** y realizar el despliegue mediante **Streamlit Community Cloud**.
